# Interfaces and Composition in C#

## Overview

In this notebook, you'll learn about four fundamental concepts in object-oriented programming:

1. **Inheritance** - Creating class hierarchies through "is-a" relationships
2. **Interfaces** - Defining contracts that classes must implement
3. **Encapsulation** - Hiding internal implementation details
4. **Composition** - Building classes from other classes through "has-a" relationships

These concepts are essential for writing maintainable, flexible, and reusable code.

## Part 1: Inheritance

**Inheritance** allows you to create a class hierarchy where a derived class inherits properties and methods from a base class. This establishes an "is-a" relationship.

### Key Points:
- Base class contains common functionality
- Derived classes extend the base class
- Promotes code reuse
- Can use `virtual` and `override` keywords to allow method overriding

In [ ]:
// Base class
public class Animal
{
    public string Name { get; set; }
    public int Age { get; set; }
    
    public Animal(string name, int age)
    {
        Name = name;
        Age = age;
    }
    
    // Virtual method that can be overridden
    public virtual void MakeSound()
    {
        Console.WriteLine($"{Name} makes a sound");
    }
    
    public void Sleep()
    {
        Console.WriteLine($"{Name} is sleeping");
    }
}

// Derived class inherits from Animal
public class Dog : Animal
{
    public string Breed { get; set; }
    
    public Dog(string name, int age, string breed) : base(name, age)
    {
        Breed = breed;
    }
    
    // Override the virtual method
    public override void MakeSound()
    {
        Console.WriteLine($"{Name} barks: Woof! Woof!");
    }
}

// Another derived class
public class Cat : Animal
{
    public Cat(string name, int age) : base(name, age) { }
    
    public override void MakeSound()
    {
        Console.WriteLine($"{Name} meows: Meow!");
    }
}

// Test inheritance
Animal dog = new Dog("Rex", 5, "Golden Retriever");
Animal cat = new Cat("Whiskers", 3);

dog.MakeSound();
cat.MakeSound();
dog.Sleep();

## Part 2: Interfaces

**Interfaces** define a contract that classes must follow. They specify what methods and properties a class should implement, but not how to implement them.

### Key Points:
- Define a set of methods and properties without implementation
- Classes implement interfaces using the `:` syntax
- A class can implement multiple interfaces (but can only inherit from one class)
- Interfaces enable loose coupling and dependency injection
- Useful for achieving polymorphism without inheritance

In [ ]:
// Define interfaces
public interface IDrivable
{
    void Start();
    void Stop();
    int GetSpeed();
}

public interface IRefuelable
{
    void Refuel(int liters);
    int GetFuelLevel();
}

// Class implementing multiple interfaces
public class Car : IDrivable, IRefuelable
{
    private string make;
    private int speed = 0;
    private int fuelLevel = 50;
    private bool isRunning = false;
    
    public Car(string make)
    {
        this.make = make;
    }
    
    // IDrivable implementation
    public void Start()
    {
        isRunning = true;
        Console.WriteLine($"{make} engine started");
    }
    
    public void Stop()
    {
        isRunning = false;
        speed = 0;
        Console.WriteLine($"{make} engine stopped");
    }
    
    public int GetSpeed() => speed;
    
    // IRefuelable implementation
    public void Refuel(int liters)
    {
        fuelLevel = Math.Min(fuelLevel + liters, 100);
        Console.WriteLine($"Refueled {liters}L. Total fuel: {fuelLevel}L");
    }
    
    public int GetFuelLevel() => fuelLevel;
}

// Test interfaces
var myCar = new Car("Toyota");
myCar.Start();
myCar.Refuel(30);
Console.WriteLine($"Fuel level: {myCar.GetFuelLevel()}L");
myCar.Stop();

### Interfaces for Polymorphism

Interfaces allow you to work with different types through a common contract:

In [ ]:
public interface IAnimal
{
    void MakeSound();
    string GetSpecies();
}

public class Dog : IAnimal
{
    public void MakeSound() => Console.WriteLine("Woof!");
    public string GetSpecies() => "Dog";
}

public class Bird : IAnimal
{
    public void MakeSound() => Console.WriteLine("Tweet!");
    public string GetSpecies() => "Bird";
}

// Function that accepts any IAnimal
void PerformAnimalActions(IAnimal animal)
{
    Console.WriteLine($"This is a {animal.GetSpecies()}");
    animal.MakeSound();
}

// Work with different types through the interface
IAnimal[] animals = new IAnimal[] { new Dog(), new Bird() };

foreach (var animal in animals)
{
    PerformAnimalActions(animal);
    Console.WriteLine();
}

## Part 3: Encapsulation

**Encapsulation** is the practice of bundling data (fields) and methods together, while hiding the internal details from the outside world. This protects the object's state and prevents unwanted modifications.

### Key Points:
- Use access modifiers: `public`, `private`, `protected`, `internal`
- Hide implementation details with `private` fields
- Expose behavior through `public` methods and properties
- Use properties with getters and setters for controlled access
- Prevents invalid state changes

In [ ]:
// Example without proper encapsulation (BAD)
public class BankAccountBad
{
    public decimal balance;  // Exposed field - anyone can modify it!
    
    public BankAccountBad(decimal initialBalance)
    {
        balance = initialBalance;
    }
}

// Example with proper encapsulation (GOOD)
public class BankAccount
{
    private decimal balance;  // Hidden field
    private string accountNumber;
    
    // Public property with validation
    public decimal Balance
    {
        get { return balance; }
        // No public setter - balance can only be changed through Deposit/Withdraw
    }
    
    public string AccountNumber
    {
        get { return accountNumber; }
        private set { accountNumber = value; }  // Only settable from within the class
    }
    
    public BankAccount(string accountNumber, decimal initialBalance)
    {
        AccountNumber = accountNumber;
        balance = initialBalance;
    }
    
    // Controlled methods to modify internal state
    public void Deposit(decimal amount)
    {
        if (amount <= 0)
            throw new ArgumentException("Deposit amount must be positive");
        balance += amount;
        Console.WriteLine($"Deposited {amount:C}. New balance: {balance:C}");
    }
    
    public void Withdraw(decimal amount)
    {
        if (amount <= 0)
            throw new ArgumentException("Withdrawal amount must be positive");
        if (amount > balance)
            throw new InvalidOperationException("Insufficient funds");
        balance -= amount;
        Console.WriteLine($"Withdrew {amount:C}. New balance: {balance:C}");
    }
}

// Test encapsulation
var account = new BankAccount("ACC123", 1000);
Console.WriteLine($"Account: {account.AccountNumber}, Balance: {account.Balance:C}");

account.Deposit(500);
account.Withdraw(200);

// This would throw an error - cannot access or modify balance directly
// account.balance = -5000;  // Not possible!
// account.Balance = 999999;  // Can't do this either!

// Instead, must use the public methods
try
{
    account.Withdraw(10000);
}
catch (InvalidOperationException ex)
{
    Console.WriteLine($"Error: {ex.Message}");
}

## Part 4: Composition

**Composition** is a design technique where a class is composed of one or more objects from other classes. It establishes a "has-a" relationship instead of an "is-a" relationship.

### Key Points:
- Build complex classes from simpler objects
- More flexible than inheritance
- Avoids the "fragile base class problem"
- Easier to change implementation
- Can change behavior at runtime

### Composition vs Inheritance:
- **Inheritance**: "A dog IS an animal"
- **Composition**: "A car HAS an engine"

In [ ]:
// Simple components
public class Engine
{
    private int horsepower;
    private bool isRunning = false;
    
    public Engine(int horsepower)
    {
        this.horsepower = horsepower;
    }
    
    public void Start()
    {
        isRunning = true;
        Console.WriteLine($"Engine started ({horsepower} HP)");
    }
    
    public void Stop()
    {
        isRunning = false;
        Console.WriteLine("Engine stopped");
    }
    
    public bool IsRunning => isRunning;
}

public class Transmission
{
    private string type;
    private int currentGear = 0;
    
    public Transmission(string type)
    {
        this.type = type;
    }
    
    public void ShiftGear(int gear)
    {
        currentGear = gear;
        Console.WriteLine($"Shifted to gear {gear} ({type} transmission)");
    }
}

public class Wheels
{
    private int wheelCount;
    private string tireType;
    
    public Wheels(int count, string tireType)
    {
        wheelCount = count;
        this.tireType = tireType;
    }
    
    public void Rotate()
    {
        Console.WriteLine($"Rotating {wheelCount} {tireType} wheels");
    }
}

// Car class COMPOSED of other objects
public class CompositeCar
{
    private string model;
    private Engine engine;           // HAS-A relationship
    private Transmission transmission; // HAS-A relationship
    private Wheels wheels;            // HAS-A relationship
    
    public CompositeCar(string model)
    {
        this.model = model;
        // Compose the car from individual components
        this.engine = new Engine(200);
        this.transmission = new Transmission("Automatic");
        this.wheels = new Wheels(4, "All-season");
    }
    
    public void Start()
    {
        engine.Start();
    }
    
    public void Drive()
    {
        if (!engine.IsRunning)
        {
            Console.WriteLine("Engine is not running. Start the car first!");
            return;
        }
        
        transmission.ShiftGear(1);
        wheels.Rotate();
        Console.WriteLine($"{model} is driving");
    }
    
    public void Stop()
    {
        wheels.Rotate();
        transmission.ShiftGear(0);
        engine.Stop();
    }
}

// Test composition
var car = new CompositeCar("Tesla Model 3");
car.Start();
car.Drive();
car.Stop();

### Composition with Dependency Injection

You can inject different implementations of components at runtime:

In [ ]:
// Component interface
public interface ILogger
{
    void Log(string message);
}

// Different implementations
public class ConsoleLogger : ILogger
{
    public void Log(string message) => Console.WriteLine($"[Console] {message}");
}

public class FileLogger : ILogger
{
    public void Log(string message) => Console.WriteLine($"[File] {message}");
}

// Service that uses composition with dependency injection
public class UserService
{
    private ILogger logger;  // Depends on abstraction, not concrete class
    
    // Constructor injection
    public UserService(ILogger logger)
    {
        this.logger = logger;
    }
    
    public void CreateUser(string userName)
    {
        logger.Log($"Creating user: {userName}");
        // ... actual user creation logic ...
        logger.Log($"User {userName} created successfully");
    }
}

// Use with different loggers
ILogger consoleLog = new ConsoleLogger();
var service1 = new UserService(consoleLog);
service1.CreateUser("Alice");

Console.WriteLine();

ILogger fileLog = new FileLogger();
var service2 = new UserService(fileLog);
service2.CreateUser("Bob");

## Part 5: Best Practices - When to Use Each Concept

### Use **Inheritance** When:
- There's a clear hierarchical relationship ("is-a")
- Multiple classes share significant common functionality
- You want to establish a contract through a base class
- **Example**: Animal → Dog, Cat, Bird

### Use **Interfaces** When:
- You want to define a contract without implementation details
- Multiple unrelated classes need to implement the same behavior
- You want to enable dependency injection and testing
- You need multiple inheritance (since a class can implement many interfaces)
- **Example**: IPaymentProcessor, IDrivable, ISerializable

### Use **Encapsulation** Always:
- To protect object state and prevent invalid modifications
- To hide internal implementation details
- To provide public interfaces while keeping internals private
- To make future refactoring easier without breaking external code

### Use **Composition** When:
- You need to combine multiple objects to create complex behavior
- You want to change behavior at runtime
- Inheritance would create overly complex hierarchies
- You want to reuse objects from different parts of your application
- **Example**: Car has Engine, Transmission, Wheels
- **Rule of Thumb**: "Favor composition over inheritance"

## Comparison Example: Inheritance vs Composition

Let's see how the same functionality can be implemented using inheritance vs composition:

In [ ]:
// ============ INHERITANCE APPROACH ============
public abstract class PaymentMethod
{
    public abstract void ProcessPayment(decimal amount);
}

public class CreditCardPayment : PaymentMethod
{
    private string cardNumber;
    
    public CreditCardPayment(string cardNumber)
    {
        this.cardNumber = cardNumber;
    }
    
    public override void ProcessPayment(decimal amount)
    {
        Console.WriteLine($"Processing credit card payment: {amount:C}");
    }
}

public class PayPalPayment : PaymentMethod
{
    private string email;
    
    public PayPalPayment(string email)
    {
        this.email = email;
    }
    
    public override void ProcessPayment(decimal amount)
    {
        Console.WriteLine($"Processing PayPal payment: {amount:C}");
    }
}

// ============ COMPOSITION APPROACH ============
public interface IPaymentProcessor
{
    void Process(decimal amount);
}

public class CreditCardProcessor : IPaymentProcessor
{
    private string cardNumber;
    
    public CreditCardProcessor(string cardNumber)
    {
        this.cardNumber = cardNumber;
    }
    
    public void Process(decimal amount)
    {
        Console.WriteLine($"Processing credit card payment: {amount:C}");
    }
}

public class PaymentGateway
{
    private IPaymentProcessor processor;  // Composition!
    
    public PaymentGateway(IPaymentProcessor processor)
    {
        this.processor = processor;  // Inject at runtime
    }
    
    public void ProcessOrder(decimal amount)
    {
        processor.Process(amount);
    }
}

// Test comparison
Console.WriteLine("=== Inheritance Approach ===");
PaymentMethod payment1 = new CreditCardPayment("1234-5678");
payment1.ProcessPayment(99.99m);

Console.WriteLine("\n=== Composition Approach ===");
IPaymentProcessor processor = new CreditCardProcessor("1234-5678");
var gateway = new PaymentGateway(processor);
gateway.ProcessOrder(99.99m);

Console.WriteLine("\n✓ Composition is more flexible and easier to test!");

## Key Takeaways

1. **Inheritance** - Use for "is-a" relationships in well-defined hierarchies
2. **Interfaces** - Use to define contracts and enable loose coupling
3. **Encapsulation** - Always use to protect object state and hide implementation
4. **Composition** - Prefer for flexibility and avoiding fragile hierarchies

### Design Principle
> **"Favor composition over inheritance"** - This means composition is often a better choice for building flexible, maintainable systems.

### When Designing Classes, Ask Yourself:
- Does this class need to share common behavior with other classes? → Consider **inheritance**
- Do I need a contract for different implementations? → Use an **interface**
- Should internal details be hidden from users of this class? → Apply **encapsulation**
- Can I break this into simpler, reusable objects? → Consider **composition**